In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
csv_path=os.path.join(path,"/kaggle/input/q1-ka-ai-2026/Q1_data.csv")
df=pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
#Drop the 'Order_ID' column from the data
df_clean=df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
#chek for null
df.isnull().sum()
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

In [ ]:
# Task 3: Write your code here:
#Check and remove duplicates if any exist
df.duplicated().sum()
#remove doplicate
df.drop_duplicates(inplace=True)

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)

from sklearn.preprocessing import OneHotEncoder
import pandas as pd
df.columns = df.columns.str.strip()
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(categorical_cols))
df = pd.concat([df.drop(columns=categorical_cols), encoded_df], axis=1)

df.head()

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# Check target imbalance
target_col = 'Delivery_Time'

class_counts = df[target_col].value_counts()
print(class_counts)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
mean_value = df['Delivery_Time'].mean()
print("Mean Delivery_Time:", mean_value)
df['Delivery_Time'] = df['Delivery_Time'].fillna(mean_value)


In [ ]:
# Task 1: Write your code here:
#Split the dataset into features (X) and target (y)
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']



In [ ]:
# Task 2,3,4,5: Write your code here:
#Use the correct split: KFold OR StratifiedKFold
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
model = RandomForestRegressor(n_estimators=100, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)
print("Average MAE across folds:", np.mean(mae_scores))

In [ ]:
# Task 1: Write your code here:
#Plot feature importance from your trained model
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
feature_importances = model.feature_importances_
feature_names = X.columns
plt.figure(figsize=(10, 6))
plt.barh(feature_names, feature_importances)
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Random Forest Feature Importance')
plt.show()

In [ ]:
# Task 2: Write your code here:
#Plot predicted delivery time histogram
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')

In [ ]:
!pip install catboost

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

In [ ]:
# Task Bonus: Write your code here:
#Why use one model? Let's use 2 then merge!
model1 = RandomForestRegressor(n_estimators=100, random_state=42)
model2 = CatBoostRegressor(verbose=0, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train both models
    model1.fit(X_train, y_train)
    model2.fit(X_train, y_train)

    # Predict
    pred1 = model1.predict(X_test)
    pred2 = model2.predict(X_test)

    # Average predictions
    avg_pred = (pred1 + pred2) / 2

    # Calculate MAE
    mae = mean_absolute_error(y_test, avg_pred)
    mae_scores.append(mae)

print("Average MAE of Rf and CatBoost:", np.mean(mae_scores))
